In [ ]:
import pyulog
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for script execution
import matplotlib.pyplot as plt
from pathlib import Path

LOG_ROOT = Path("../px4_logs")

In [ ]:
ulg_files = sorted(LOG_ROOT.rglob("*.ulg"))
print(f"Found {len(ulg_files)} .ulg files")

rows = []
for p in ulg_files:
    try:
        log = pyulog.ULog(str(p), disable_str_exceptions=True)
        topics = {d.name for d in log.data_list}
        rows.append({"path": str(p.relative_to(LOG_ROOT)), "topics": len(topics),
                     "has_actuator": "actuator_outputs" in topics,
                     "has_imu": "sensor_combined" in topics,
                     "has_rates": "vehicle_angular_velocity" in topics,
                     "has_position": "vehicle_local_position" in topics,
                     "has_attitude": "vehicle_attitude" in topics,
                     "has_rate_sp": "vehicle_rates_setpoint" in topics,
                     "has_pos_sp": "vehicle_local_position_setpoint" in topics,
                     "duration_s": (log.last_timestamp - log.start_timestamp) / 1e6})
    except Exception as e:
        rows.append({"path": str(p.relative_to(LOG_ROOT)), "error": str(e)})

df = pd.DataFrame(rows)
df

In [ ]:
bool_cols = ["has_actuator", "has_imu", "has_rates", "has_position",
             "has_attitude", "has_rate_sp", "has_pos_sp"]
heat = df[bool_cols].fillna(False).astype(float)
fig, ax = plt.subplots(figsize=(10, max(4, len(heat) * 0.3)))
im = ax.imshow(heat.T, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_yticks(range(len(bool_cols))); ax.set_yticklabels(bool_cols)
ax.set_xlabel("Log index"); ax.set_title("Signal availability across all .ulg files")
plt.tight_layout(); plt.savefig("/tmp/heatmap.png"); plt.close()
print("Saved heatmap to /tmp/heatmap.png")

In [ ]:
LOG_PATH = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Actuator outputs
try:
    act = to_df(log, "actuator_outputs")
    for i in range(4):
        axes[0].plot(act["t_s"], act[f"output[{i}]"], label=f"motor {i+1}")
    axes[0].set_title("Actuator Outputs"); axes[0].legend()
except Exception as e:
    axes[0].set_title(f"Actuator Outputs (unavailable: {e})")

# Angular velocity
try:
    rates = to_df(log, "vehicle_angular_velocity")
    for ax_name, col in [("roll", "xyz[0]"), ("pitch", "xyz[1]"), ("yaw", "xyz[2]")]:
        axes[1].plot(rates["t_s"], np.rad2deg(rates[col]), label=ax_name)
    axes[1].set_title("Angular Velocity (deg/s)"); axes[1].legend()
except Exception as e:
    axes[1].set_title(f"Angular Velocity (unavailable: {e})")

# Attitude
try:
    att = to_df(log, "vehicle_attitude")
    from scipy.spatial.transform import Rotation as R_
    q = att[["q[0]", "q[1]", "q[2]", "q[3]"]].values  # [w, x, y, z]
    euler = R_.from_quat(q[:, [1, 2, 3, 0]]).as_euler("xyz", degrees=True)
    for i, name in enumerate(["roll", "pitch", "yaw"]):
        axes[2].plot(att["t_s"].values, euler[:, i], label=name)
    axes[2].set_title("Attitude (deg)"); axes[2].legend()
except Exception as e:
    axes[2].set_title(f"Attitude (unavailable: {e})")

# Local position
try:
    pos = to_df(log, "vehicle_local_position")
    for col, name in [("x", "X"), ("y", "Y"), ("z", "Z")]:
        axes[3].plot(pos["t_s"], pos[col], label=name)
    axes[3].set_title("Local Position (m)"); axes[3].legend(); axes[3].set_xlabel("Time (s)")
except Exception as e:
    axes[3].set_title(f"Local Position (unavailable: {e})"); axes[3].set_xlabel("Time (s)")

plt.tight_layout(); plt.savefig("/tmp/signals.png"); plt.close()
print("Saved signal plot to /tmp/signals.png")

In [ ]:
print("=== PID Parameters from log ===")
pid_params = {k: v for k, v in log.initial_parameters.items()
              if any(x in k for x in ["MC_ROLL", "MC_PITCH", "MC_YAW", "MPC_XY", "MPC_Z_"])}
for k in sorted(pid_params):
    print(f"  {k}: {pid_params[k]:.6f}")